### Data Prep

In [0]:
%sql
CREATE EXTERNAL TABLE IF NOT EXISTS kafka_poc.engagement.evolution_demo_external (
  `partition` INT,
  `offset` BIGINT,
  post_id STRING,
  `type` STRING,
  user_id STRING,
  ts BIGINT,
  device STRING,
  country STRING)
USING DELTA
LOCATION 's3://retail-data-lake-dev/delta_poc/data/evolution_demo/'
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true',
  'delta.enableDeletionVectors' = 'true',
  'delta.enableRowTracking'     = 'true',
  'delta.parquet.compression.codec' = 'zstd')

In [0]:
%sql
MERGE INTO kafka_poc.engagement.evolution_demo_external AS t
USING (
  SELECT * FROM VALUES
    (0, 1001, 'p001', 'like',    'u01', 1733001000, 'ios',     'US'),
    (0, 1002, 'p002', 'view',    'u02', 1733001050, 'android', 'IN'),
    (1, 1003, 'p003', 'comment', 'u03', 1733001100, 'web',     'GB'),
    (1, 1004, 'p004', 'share',   'u04', 1733001150, 'ios',     'US'),
    (2, 1005, 'p005', 'like',    'u05', 1733001200, 'android', 'DE'),
    (2, 1006, 'p006', 'click',   'u06', 1733001250, 'web',     'IN'),
    (0, 1007, 'p007', 'view',    'u07', 1733001300, 'ios',     'FR'),
    (1, 1008, 'p008', 'like',    'u08', 1733001350, 'android', 'US'),
    (2, 1009, 'p009', 'comment', 'u09', 1733001400, 'web',     'IN'),
    (0, 1010, 'p010', 'share',   'u10', 1733001450, 'ios',     'GB')
  AS s(`partition`, `offset`, post_id, type, user_id, ts, device, country)
) AS s
ON t.post_id = s.post_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

In [0]:
%sql
MERGE INTO kafka_poc.engagement.evolution_demo_external AS t
USING (
  SELECT * FROM VALUES
    -- 5 that will MATCH p001..p005 (changed type/offset/ts/device/country)
    (0, 2001, 'p001', 'share',   'u01', 1733005000, 'web',     'US'),
    (0, 2002, 'p002', 'like',    'u02', 1733005050, 'ios',     'IN'),
    (1, 2003, 'p003', 'view',    'u03', 1733005100, 'android', 'GB'),
    (1, 2004, 'p004', 'comment', 'u04', 1733005150, 'web',     'US'),
    (2, 2005, 'p005', 'click',   'u05', 1733005200, 'ios',     'DE'),
    -- 5 that will NOT MATCH p011..p015 (new)
    (2, 2011, 'p011', 'like',    'u11', 1733005250, 'android', 'IN'),
    (0, 2012, 'p012', 'view',    'u12', 1733005300, 'web',     'US'),
    (1, 2013, 'p013', 'comment', 'u13', 1733005350, 'ios',     'FR'),
    (2, 2014, 'p014', 'share',   'u14', 1733005400, 'android', 'GB'),
    (0, 2015, 'p015', 'click',   'u15', 1733005450, 'web',     'IN')
  AS s(`partition`, `offset`, post_id, type, user_id, ts, device, country)
) AS s
ON t.post_id = s.post_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

In [0]:
%sql
desc history kafka_poc.engagement.evolution_demo_external

In [0]:
%fs
ls s3://retail-data-lake-dev/delta_poc/data/evolution_demo/

In [0]:
%fs
ls s3://retail-data-lake-dev/delta_poc/data/evolution_demo/_delta_log/

In [0]:
%fs
ls s3://retail-data-lake-dev/delta_poc/data/evolution_demo/_change_data/

In [0]:
spark.read.parquet('s3://retail-data-lake-dev/delta_poc/data/evolution_demo/_change_data/cdc-00000-f2d6e3d9-51bc-4b5b-8947-b250924c6f9e.c000.zstd.parquet').display()